### **SPARK STREAMING**

In [0]:
entities =['customers','trips','payments','drivers','locations','vehicles']

In [0]:
for entity in entities:

    # Batch read to infer schema
    df_batch = spark.read.format("csv") \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .load(f"/Volumes/pysparkdbt/source/source_data/{entity}/")

    schema_entity = df_batch.schema

    # Streaming read
    df = spark.readStream.format("csv") \
        .option("header", "true") \
        .schema(schema_entity) \
        .load(f"/Volumes/pysparkdbt/source/source_data/{entity}/")

    # Write stream
    df.writeStream.format("delta") \
        .outputMode("append") \
        .option("checkpointLocation", f"/Volumes/pysparkdbt/bronze/checkpoint/{entity}") \
        .trigger(once=True) \
        .toTable(f"pysparkdbt.bronze.{entity}")